# jobs: BALLS64

host = ```any```, device = ```any```

**Motivation**: <br>

Create jobs for BALLS dataset.

- mach: Gaussians (None + ReLU)
- yoru: Poisson

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

project_name = '_IterativeVAE'

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, project_name))
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

## Setup

In [2]:
from base.helper import job_runner_script


def _cleanup(path, host=None):
    for f in os.listdir(path):
        cond = f.endswith('.txt')
        if host is not None:
            cond = cond and host in f
        if cond:
            os.remove(pjoin(path, f))


def _name(host, gpu_i, fit_i):
    return f"{host}-cuda{gpu_i}-fit{fit_i}"

In [3]:
save_dir = f'Dropbox/git/{project_name}/scripts'
save_dir = pjoin(os.environ['HOME'], save_dir)
os.makedirs(save_dir, exist_ok=True)

# delete existing job runners?
_cleanup(save_dir, None)

print(sorted(os.listdir(save_dir)))

[
    'cleanup_chkpts.sh',
    'cleanup_recursive.sh',
    'copyfits.sh',
    'fit_model.sh',
    'kill_screens.sh',
    'resume_fit.sh',
    'run_sessions.sh',
    'test_tqdm.py',
    'test_tqdm.sh'
]

## Gaussian: None

### mach

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(10, 20)

model_type = 'gaussian'

t_train = 32
beta_outer = 1.0
n_latents = 96
dataset = 'BALLS64'

seeds

array([10, 11, 12, 13, 14, 15, 16, 17, 18, 19])

In [6]:
tot = 0

for seed in seeds:
    # for latent_act in latent_act_list:
    arg = [
        f"--t_train {t_train}",
        f"--beta_outer {beta_outer}",
        f"--n_latents {n_latents}",
        # f"--latent_act '{latent_act}'" if latent_act else '',
        '--verbose',
    ]
    arg = ' '.join(filter(None, arg))

    gpu_i = tot % torch.cuda.device_count()
    gpu_i = (gpu_i + 2) % 4

    kws = dict(
        device=gpu_i,
        dataset=dataset,
        model=model_type,
        args=arg,
        seed=seed,
    )
    scripts[gpu_i].append(job_runner_script(**kws))
    tot += 1

In [7]:
print(tot)

10

In [8]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{0: 2, 1: 2, 2: 3, 3: 3}

#### Save

In [9]:
n_fits = 3

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
            verbose=False,
        )
        print(combined.replace('&& ', '&& \n'))

./fit_model.sh '0' 'BALLS64' 'gaussian' --seed 12 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '0' 'BALLS64' 'gaussian' --seed 16 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '1' 'BALLS64' 'gaussian' --seed 13 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '1' 'BALLS64' 'gaussian' --seed 17 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '2' 'BALLS64' 'gaussian' --seed 10 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '2' 'BALLS64' 'gaussian' --seed 14 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '2' 'BALLS64' 'gaussian' --seed 18 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '3' 'BALLS64' 'gaussian' --seed 11 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '3' 'BALLS64' 'gaussian' --seed 15 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

./fit_model.sh '3' 'BALLS64' 'gaussian' --seed 19 --t_train 32 --beta_outer 1.0 --n_latents 96 --verbose

## Gaussian: ReLU

### chewie

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = [0, 1] # np.arange(10, 20)

model_type = 'gaussian'
latent_act = 'relu'

t_train_list = [32]
beta_outer = 1.0
n_latents = 96
dataset = 'BALLS64'

seeds

[0, 1]

In [6]:
tot = 0

for seed in seeds:
    for t_train in t_train_list:
        arg = [
            f"--t_train {t_train}",
            f"--beta_outer {beta_outer}",
            f"--n_latents {n_latents}",
            f"--latent_act '{latent_act}'" if latent_act else '',
            '--verbose',
        ]
        arg = ' '.join(filter(None, arg))
    
        gpu_i = tot % torch.cuda.device_count()
    
        kws = dict(
            device=gpu_i,
            dataset=dataset,
            model=model_type,
            args=arg,
            seed=seed,
        )
        scripts[gpu_i].append(job_runner_script(**kws))
        tot += 1

In [7]:
print(tot)

2

In [8]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{0: 1, 1: 1}

#### Save

In [9]:
n_fits = 1

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
            verbose=False,
        )
        print(combined.replace('&& ', '&& \n'))

./fit_model.sh '0' 'BALLS64' 'gaussian' --seed 0 --t_train 32 --beta_outer 1.0 --n_latents 96 --latent_act 'relu' 
--verbose

./fit_model.sh '1' 'BALLS64' 'gaussian' --seed 1 --t_train 32 --beta_outer 1.0 --n_latents 96 --latent_act 'relu' 
--verbose

## Poisson

### yoru

In [4]:
host = 'yoru'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(500, 502)

model_type = 'poisson'

t_train = 16
beta_outer = 24.0
clamp_u_list = [10.0, 6.0, 4.5]
learning_rates = [5e-4]
n_latents = 128
dataset = 'BALLS64'

seeds

array([500, 501])

In [6]:
tot = 0

# for gpu_i, seed in enumerate(seeds, start=0):
for seed in seeds:
    for lr in learning_rates:
        for clamp_u in clamp_u_list:
            arg = [
                f"--t_train {t_train}",
                f"--beta_outer {beta_outer}",
                f"--n_latents {n_latents}",
                f'--lr {lr}',
                f'--clamp_u {clamp_u}',
                f'--comment lr-{lr}_cpois-{clamp_u}_cdec-{2.3}',
                '--verbose',
            ]
            arg = ' '.join(filter(None, arg))

            # gpu_i = tot % torch.cuda.device_count()
            # gpu_i = tot % 2
            gpu_i = 1
        
            kws = dict(
                device=gpu_i,
                dataset=dataset,
                model=model_type,
                args=arg,
                seed=seed,
            )
            scripts[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

6

In [8]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{1: 6}

#### Save

In [9]:
n_fits = 3

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
            verbose=False,
        )
        print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'BALLS64' 'poisson' --seed 500 --t_train 16 --beta_outer 24.0 --n_latents 128 --lr 0.0005 
--clamp_u 10.0 --comment lr-0.0005_cpois-10.0_cdec-2.3 --verbose && 
./fit_model.sh '1' 'BALLS64' 'poisson' --seed 500 --t_train 16 --beta_outer 24.0 --n_latents 128 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-2.3 --verbose

./fit_model.sh '1' 'BALLS64' 'poisson' --seed 500 --t_train 16 --beta_outer 24.0 --n_latents 128 --lr 0.0005 
--clamp_u 4.5 --comment lr-0.0005_cpois-4.5_cdec-2.3 --verbose && 
./fit_model.sh '1' 'BALLS64' 'poisson' --seed 501 --t_train 16 --beta_outer 24.0 --n_latents 128 --lr 0.0005 
--clamp_u 10.0 --comment lr-0.0005_cpois-10.0_cdec-2.3 --verbose

./fit_model.sh '1' 'BALLS64' 'poisson' --seed 501 --t_train 16 --beta_outer 24.0 --n_latents 128 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-2.3 --verbose && 
./fit_model.sh '1' 'BALLS64' 'poisson' --seed 501 --t_train 16 --beta_outer 24.0 --n_latents 128 --lr 0.0005 
--clamp_u 4.5 --comment lr-0.0005_cpois-4.5_cdec-2.3 --verbose

### chewie

In [4]:
host = 'chewie'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(503, 505)

model_type = 'poisson'

t_train = 32
beta_outer = 96.0
clamp_u_list = [6.0]
learning_rates = [5e-4]
n_latents = 96
dataset = 'BALLS64'

seeds

array([503, 504])

In [6]:
tot = 0

for gpu_i, seed in enumerate(seeds, start=0):
# for seed in seeds:
    for lr in learning_rates:
        for clamp_u in clamp_u_list:
            arg = [
                f"--t_train {t_train}",
                f"--beta_outer {beta_outer}",
                f"--n_latents {n_latents}",
                f'--lr {lr}',
                f'--clamp_u {clamp_u}',
                f'--comment lr-{lr}_cpois-{clamp_u}_cdec-{3.45}',
                '--verbose',
            ]
            arg = ' '.join(filter(None, arg))

            # gpu_i = tot % torch.cuda.device_count()
            # gpu_i = tot % 2
        
            kws = dict(
                device=gpu_i,
                dataset=dataset,
                model=model_type,
                args=arg,
                seed=seed,
            )
            scripts[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

2

In [8]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{0: 1, 1: 1}

#### Save

In [9]:
n_fits = 1

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
            verbose=False,
        )
        print(combined.replace('&& ', '&& \n'))

./fit_model.sh '0' 'BALLS64' 'poisson' --seed 503 --t_train 32 --beta_outer 96.0 --n_latents 96 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-3.45 --verbose

./fit_model.sh '1' 'BALLS64' 'poisson' --seed 504 --t_train 32 --beta_outer 96.0 --n_latents 96 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-3.45 --verbose

### mach

In [4]:
host = 'mach'
_cleanup(save_dir, host)

scripts = collections.defaultdict(list)
tot = 0

In [5]:
seeds = np.arange(500, 503)

model_type = 'poisson'

t_train = 32
beta_outer = 96.0
clamp_u_list = [6.0]
learning_rates = [5e-4]
n_latents = 96
dataset = 'BALLS64'

seeds

array([500, 501, 502])

In [6]:
tot = 0

for gpu_i, seed in enumerate(seeds, start=1):
# for seed in seeds:
    for lr in learning_rates:
        for clamp_u in clamp_u_list:
            arg = [
                f"--t_train {t_train}",
                f"--beta_outer {beta_outer}",
                f"--n_latents {n_latents}",
                f'--lr {lr}',
                f'--clamp_u {clamp_u}',
                f'--comment lr-{lr}_cpois-{clamp_u}_cdec-{3.45}',
                '--verbose',
            ]
            arg = ' '.join(filter(None, arg))

            # gpu_i = tot % torch.cuda.device_count()
            # gpu_i = tot % 2
        
            kws = dict(
                device=gpu_i,
                dataset=dataset,
                model=model_type,
                args=arg,
                seed=seed,
            )
            scripts[gpu_i].append(job_runner_script(**kws))
            tot += 1

In [7]:
print(tot)

3

In [8]:
scripts = dict(sorted(scripts.items()))
print({k: len(v) for k, v in scripts.items()})

{1: 1, 2: 1, 3: 1}

#### Save

In [9]:
n_fits = 1

for gpu_i, scripts_list in scripts.items():
    scripts_divided = divide_list(scripts_list, n_fits)
    for fit_i, s in enumerate(scripts_divided):
        combined = ' && '.join(s)
        save_obj(
            obj=combined,
            file_name=_name(host, gpu_i, fit_i),
            save_dir=save_dir,
            mode='txt',
            verbose=False,
        )
        print(combined.replace('&& ', '&& \n'))

./fit_model.sh '1' 'BALLS64' 'poisson' --seed 500 --t_train 32 --beta_outer 96.0 --n_latents 96 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-3.45 --verbose

./fit_model.sh '2' 'BALLS64' 'poisson' --seed 501 --t_train 32 --beta_outer 96.0 --n_latents 96 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-3.45 --verbose

./fit_model.sh '3' 'BALLS64' 'poisson' --seed 502 --t_train 32 --beta_outer 96.0 --n_latents 96 --lr 0.0005 
--clamp_u 6.0 --comment lr-0.0005_cpois-6.0_cdec-3.45 --verbose